In [5]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import time
import warnings

import joblib
import numpy as np
import polars as pl
import tensorflow as tf
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = 'label'
N_REPEATS = 3
WARMUP_SIZE = 1000


In [6]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para IOT-23
prepared_path = '../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv'

df_encoded = pl.read_csv(prepared_path)

feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy().astype(np.float32)

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]

X_full_base = X_np.astype(np.float32)

mlp_scaler = StandardScaler()
mlp_scaler.fit(X_full_train_np)
X_full_scaled_mlp = mlp_scaler.transform(X_full_base).astype(np.float32)

cnn_scaler = MinMaxScaler()
cnn_scaler.fit(X_full_train_np)
X_full_scaled_cnn = cnn_scaler.transform(X_full_base).astype(np.float32)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)

N_MUESTRAS = len(X_full_base)

print(f'Train full: {len(X_full_train_np):,} muestras')
print(f'Test:       {len(X_test_np):,} muestras')
print(f'Total:      {N_MUESTRAS:,} muestras')
print(f'Features:   {X_full_base.shape[1]}')


Train full: 931,880 muestras
Test:       232,971 muestras
Total:      1,164,851 muestras
Features:   22


In [7]:
# RF

rf_path = '../model/iot-23/rf_iot23.joblib'

rf_model = joblib.load(rf_path)
print('Modelo cargado')

# Warm-up
_ = rf_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))
throughput_rf = N_MUESTRAS / tiempo_total_rf

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s')
print(f'Throughput aproximado: {throughput_rf:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.2078, 0.206, 0.2011]
Tiempo medio de inferencia sobre todo el dataset: 0.2049 s
Throughput aproximado: 5,684,020 muestras/s


In [8]:
# XGBOOST

xgb_path = '../model/iot-23/xgb_iot23.joblib'

xgb_model = joblib.load(xgb_path)
print('Modelo cargado')

# Warm-up
_ = xgb_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))
throughput_xgb = N_MUESTRAS / tiempo_total_xgb

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s')
print(f'Throughput aproximado: {throughput_xgb:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.0254, 0.0237, 0.0235]
Tiempo medio de inferencia sobre todo el dataset: 0.0242 s
Throughput aproximado: 48,100,521 muestras/s


/usr/lib/python3.11/pickle.py:1718: UserWarning: [11:28:51] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.11/pickle.py:1718: UserWarning: [11:28:51] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.11/pickle.py:1718: UserWarning: [11:28:51] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


In [9]:
# LIGHTGBM

lgbm_path = '../model/iot-23/lgbm_iot23.joblib'

lgbm_model = joblib.load(lgbm_path)
print('Modelo cargado')

# Warm-up
_ = lgbm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))
throughput_lgbm = N_MUESTRAS / tiempo_total_lgbm

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s')
print(f'Throughput aproximado: {throughput_lgbm:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.0852, 0.0845, 0.2518]
Tiempo medio de inferencia sobre todo el dataset: 0.1405 s
Throughput aproximado: 8,290,807 muestras/s


In [10]:
# CATBOOST

catboost_path = '../model/iot-23/catboost_iot23.joblib'

catboost_model = joblib.load(catboost_path)
print('Modelo cargado')

# Warm-up
_ = catboost_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))
throughput_catboost = N_MUESTRAS / tiempo_total_catboost

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s')
print(f'Throughput aproximado: {throughput_catboost:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.3061, 0.265, 0.2609]
Tiempo medio de inferencia sobre todo el dataset: 0.2773 s
Throughput aproximado: 4,200,133 muestras/s


In [11]:
# SVM

svm_path = '../model/iot-23/svm_iot23.joblib'

svm_model = joblib.load(svm_path)
print('Modelo cargado')

# Warm-up
_ = svm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))
throughput_svm = N_MUESTRAS / tiempo_total_svm

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s')
print(f'Throughput aproximado: {throughput_svm:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.1151, 0.1367, 0.11]
Tiempo medio de inferencia sobre todo el dataset: 0.1206 s
Throughput aproximado: 9,656,967 muestras/s


In [12]:
# MLP

mlp_path = '../model/iot-23/mlp_iot23.joblib'

mlp_model = joblib.load(mlp_path)
print('Modelo cargado')

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(WARMUP_SIZE, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))
throughput_mlp = N_MUESTRAS / tiempo_total_mlp

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s')
print(f'Throughput aproximado: {throughput_mlp:,.0f} muestras/s')


E0000 00:00:1779874141.918159 1173273 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1779874141.918198 1173273 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1779874141.918209 1173273 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1779874141.918220 1173273 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1779874141.918222 1173273 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: ai-server1
I0000 00:00:1779874141.918225 1173273 cuda_diagnostics.cc:183] hostname: ai-server1
I0000 00:00:1779874141.918320 1173273 cuda_diagnostics.cc:190] libcuda reported version is: 580.95.5
I0000 00:00:1779874141.918336 1173273 cuda_diagnostics.cc:194] kernel r

Modelo cargado


I0000 00:00:1779874142.184171 1175218 service.cc:153] XLA service 0x7fe6a8693e90 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779874142.184206 1175218 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1779874142.212578 1175218 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779874142.388950 1175218 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Tiempos medidos: [0.6546, 0.4459, 0.4333]
Tiempo medio de inferencia sobre todo el dataset: 0.5113 s
Throughput aproximado: 2,278,408 muestras/s


In [13]:
# CNN

cnn_path = '../model/iot-23/cnn_iot23.joblib'

cnn_model = joblib.load(cnn_path)
print('Modelo cargado')

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(WARMUP_SIZE, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))
throughput_cnn = N_MUESTRAS / tiempo_total_cnn

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s')
print(f'Throughput aproximado: {throughput_cnn:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [2.0325, 1.7627, 1.7589]
Tiempo medio de inferencia sobre todo el dataset: 1.8514 s
Throughput aproximado: 629,180 muestras/s


In [14]:
# Tabla comparativa final

tabla_comparativa = pd.DataFrame(
    [
        {'Modelo': 'RF', 'Tiempo medio (s)': tiempo_total_rf, 'Muestras/s aprox': throughput_rf},
        {'Modelo': 'XGBoost', 'Tiempo medio (s)': tiempo_total_xgb, 'Muestras/s aprox': throughput_xgb},
        {'Modelo': 'LightGBM', 'Tiempo medio (s)': tiempo_total_lgbm, 'Muestras/s aprox': throughput_lgbm},
        {'Modelo': 'CatBoost', 'Tiempo medio (s)': tiempo_total_catboost, 'Muestras/s aprox': throughput_catboost},
        {'Modelo': 'SVM', 'Tiempo medio (s)': tiempo_total_svm, 'Muestras/s aprox': throughput_svm},
        {'Modelo': 'MLP', 'Tiempo medio (s)': tiempo_total_mlp, 'Muestras/s aprox': throughput_mlp},
        {'Modelo': 'CNN', 'Tiempo medio (s)': tiempo_total_cnn, 'Muestras/s aprox': throughput_cnn},
    ]
).sort_values('Tiempo medio (s)').reset_index(drop=True)

tabla_comparativa['Tiempo medio (s)'] = tabla_comparativa['Tiempo medio (s)'].round(4)
tabla_comparativa['Muestras/s aprox'] = tabla_comparativa['Muestras/s aprox'].round(0).astype(int)

display(tabla_comparativa)


,Modelo,Tiempo medio (s),Muestras/s aprox
0,XGBoost,0.0242,48100521
1,SVM,0.1206,9656967
2,LightGBM,0.1405,8290807
3,RF,0.2049,5684020
4,CatBoost,0.2773,4200133
5,MLP,0.5113,2278408
6,CNN,1.8514,629180
